# video-base-imagem-versiculo-trilhas-efeitos.ipynb — Narrated Video (cena + clima + efeito pontual)

Acumula tudo de `video-base-imagem-versiculo-trilhas.ipynb` (match
cena↔versículo, vídeo base, trilha por evento/clima) **+ efeito sonoro
pontual** (porta, trovão, cavalo...) sobreposto no momento exato do
versículo que casou com ele — pra mudanças rápidas de clima que a
trilha (que muda só quando o evento muda) não cobre.

Segundo match, independente dos outros dois: usa as mesmas
palavras-chave por versículo já calculadas no match de cena
(`tags_semelhantes`, ver `match_pipeline.registrar_tags_versiculo`)
contra um pool pequeno de efeitos candidatos que você escolhe à mão
(`EFEITOS_CANDIDATOS` na Configuração, ids/urls da aba `efeitos_stock`).
Casa por versículo individual (não por trecho de evento como a trilha)
e a maioria dos versículos fica sem efeito nenhum — é pontual de
propósito.

Se você só quer o vídeo com trilha, sem efeito sonoro, use
`video-base-imagem-versiculo-trilhas.ipynb` — é mais simples. Este
aqui é o próximo degrau da série.

**Como usar:**
1. Rode SETUP, CONFIGURAÇÃO e (se tiver) SCRIPT/ÁUDIO DO DRIVE.
2. Rode só NARRAÇÃO primeiro.
3. Rode `caption-single-generate.ipynb` separadamente (Whisper sobre a
   narração) — continua sendo pré-requisito, não foi incorporado aqui.
4. Volte aqui e rode da célula de MATCH em diante.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — pacotes, Google Drive, módulos e autenticação (1x por sessão) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Pacotes de sistema ────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# ── Pacotes Python ─────────────────────────────────────────────────────────
!pip install -q edge-tts pandas gdown yt-dlp nest_asyncio gspread requests groq "mistralai>=1.2.0"
print('✅ Pacotes Python')

# ── Monta o Drive (desmonta antes, evita sessão travada) ──────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

# ── Copia os módulos do Drive pra /content/pipeline ────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixo pro projeto inteiro (mesmo valor da Configuração)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados de {PASTA_MODULOS}")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')

# ── Autenticação (Google Sheets + IA) — precisa pro match cena/trilha ─────
from google.colab import auth, userdata
from google.auth import default
import gspread
from groq import Groq
from mistralai.client import Mistral

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None

print('✅ Setup completo!')
print(f"   Groq:    {'disponível' if groq_client else '⚠️  GROQ_KEY não encontrada (só léxico/biblioteca resolvem)'}")
print(f"   Mistral: {'disponível' if mistral_client else '⚠️  MISTRAL_KEY não encontrada'}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO                                                ║
# ║  ✏️  Edite só esta célula ao começar um vídeo novo                ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. IDENTIDADE DO VÍDEO ──────────────────────────────────────────────────
NOME_ORACAO = "40_Matt_02"           # identificador curto, sem espaço/acento
LIVRO_PT = "Mateus"                  # EXATAMENTE como aparece no léxico (titulos/eventos-biblicos.json)
CAPITULO = 2

# ── 2. TEXTO COMPLETO (Edge TTS) — ignorado se já existir roteiro no Drive ──
TEXTO_ORACAO = (
"Now when Jesus was born in Bethlehem of Judea, in the days of King Herod. "
"Behold, wise men from the east came to Jerusalem. "
"Saying, Where is He who is born King of the Jews? "
"For we saw His star in the east and have come to worship Him. "
"When King Herod heard it, he was troubled. "
"And all Jerusalem with him. "
"Gathering together all the chief priests and scribes of the people. "
"He asked them where the Christ would be born. "
"They said to him: In Bethlehem of Judea, for thus it is written through the prophet. "
"You, Bethlehem, land of Judah, are in no way least among the princes of Judah. "
"For out of you shall come forth a governor who shall shepherd My people Israel. "
"Then Herod secretly called the wise men. "
"And learned from them exactly what time the star appeared. "
"He sent them to Bethlehem and said: Go and search diligently for the young child. "
"When you have found him, bring me word, so that I also may come and worship him. "
"They, having heard the King, went their way. "
"And behold, the star which they saw in the east went before them. "
"Until it came and stood over where the young child was. "
"When they saw the star, they rejoiced with exceedingly great joy. "
"They came into the house and saw the young child with Mary, his mother. "
"And they fell down and worshiped him. "
"Opening their treasures, they offered to him gifts: gold, frankincense, and myrrh. "
"Being warned in a dream not to return to Herod. "
"They went back to their own country another way. "
"Now when they had departed, behold, an angel of the Lord appeared to Joseph in a dream. "
"Saying, Arise and take the young child and his mother. "
"And flee into Egypt and stay there until I tell you. "
"For Herod will seek the young child to destroy him. "
"He arose and took the young child and his mother by night. "
"And departed into Egypt. "
"And was there until the death of Herod. "
"That it might be fulfilled which was spoken by the Lord through the prophet. "
"Out of Egypt I called My Son. "
"Then Herod, when he saw that he was mocked by the wise men, was exceedingly angry. "
"And sent out and killed all the male children who were in Bethlehem. "
"And in all the surrounding countryside, from two years old and under. "
"According to the exact time which he had learned from the wise men. "
"Then that which was spoken by Jeremiah the prophet was fulfilled. "
"Saying, A voice was heard in Rama, lamentation, weeping, and great mourning. "
"Rachel weeping for her children, and she would not be comforted. "
"Because they are no more. "
"But when Herod was dead, behold, an angel of the Lord appeared in a dream to Joseph in Egypt. "
"Saying, Arise and take the young child and his mother. "
"And go into the land of Israel. "
"For those who sought the young child's life are dead. "
"He arose and took the young child and his mother. "
"And came into the land of Israel. "
"But when he heard that Archelaus was reigning over Judea in the place of his father Herod. "
"He was afraid to go there. "
"Being warned in a dream, he withdrew into the region of Galilee. "
"And came and lived in a city called Nazareth. "
"That it might be fulfilled which was spoken through the prophets. "
"- He will be called a Nazarene! "
)

# ── 3. VOZ DO NARRADOR ──────────────────────────────────────────────────────
#    🇧🇷 Português:  "pt-BR-AntonioNeural" (m) · "pt-BR-FranciscaNeural" (f)
#    🇺🇸 Inglês:      "en-US-GuyNeural" (m) · "en-US-JennyNeural" (f)
#    🇪🇸 Espanhol:    "es-ES-AlvaroNeural" (m) · "es-ES-ElviraNeural" (f)
VOZ_EDGE = "en-US-GuyNeural"

# ── 3b. IDIOMA MESTRE ────────────────────────────────────────────────────
# Idioma de TEXTO_ORACAO/VOZ_EDGE acima -- tem que bater. Usado pra achar o
# SRT do Whisper certo depois (caption-single-generate.ipynb).
IDIOMA_MESTRE = "en"

# ── 4. VOLUME — proporção narração x trilha ─────────────────────────────────
VOLUME_NARRACAO = 1.0    # volume da narração (1.0 = original)
VOLUME_MUSICA   = 0.25   # volume da trilha, relativo à narração

# ── 5. VELOCIDADE DA NARRAÇÃO ────────────────────────────────────────────
VELOCIDADE_AUDIO = 1.0   # 1.0 = original; 0.9 = 10% mais devagar; 1.1 = 10% mais rápido

# ── 6. MATCH CENA↔VERSÍCULO ──────────────────────────────────────────────
TIPO_FONTE = "imagem"    # "imagem" (foco atual) ou "video"

ID_PLANILHA_VIDEOS = "1bF7hnGSY7AALm4ZAS5owWNpiSTdgArW4ahAuVZaHPL0"
NOME_ABA_VIDEOS = "pixabay_stock"

ID_PLANILHA_IMAGENS = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"
NOME_ABA_IMAGENS = "image-stock"
NOME_COLUNA_STATUS_PLANILHA = "Downloading Ok"

# léxico bíblico -- primeira tentativa de tag, antes de IA (grátis, instantâneo)
USAR_LEXICO_BIBLICO = True
NOME_ARQUIVO_TITULOS = "titulos-biblicos.json"
NOME_ARQUIVO_EVENTOS = "eventos-biblicos.json"
PASTA_DADOS_LEXICO = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/dados_lexico"

# biblioteca de match (cache reutilizável entre vídeos -- mesma planilha
# guarda biblioteca_match, versiculo_tags, evento_tags, titulo_tags)
USAR_BIBLIOTECA_MATCH = True
ID_PLANILHA_BIBLIOTECA_MATCH = "1i67VxksAkWYx1cZ_QeoesGXsW28hcA0p5IIfhjx8VHE"
NOME_ABA_BIBLIOTECA_MATCH = "biblioteca_match"
USAR_VERSICULO_TAGS = True
NOME_ABA_VERSICULO_TAGS = "versiculo_tags"

# modelos de IA (texto só)
MODELO_GROQ = "qwen/qwen3.6-27b"
MODELO_MISTRAL = "mistral-small-latest"
DELAY_SEGUNDOS = 2
MAX_TOKENS_RESPOSTA = 300

# anti-repetição (não repete o mesmo vídeo/imagem em versículos muito próximos)
DIST_MIN_REPETICAO = 3
MARGEM_PARA_REPETIR = 2

# ── 7. TRILHA — pool de candidatas (match por clima, por evento) ────────────
ID_PLANILHA_BIBLIOTECA_MATCH_AUDIO = "1VkYaApN1F7X4-52CD0_I-TpKwc2v3XuhUJrIRnZ0Q94"
NOME_ABA_TRILHA_STOCK = "trilha_stock"

# Cole aqui o id OU url_preview de cada linha da aba trilha_stock que você
# quer disponibilizar como candidata -- o notebook escolhe sozinho qual
# delas serve pra cada trecho (evento) do vídeo, só entre as que estão aqui.
# Vários versículos seguidos do mesmo evento ficam sob a mesma trilha.
TRILHAS_CANDIDATAS = [
    "513984",
]

# ── 7b. EFEITOS SONOROS — pool de candidatos (match pontual, por versículo) ─
NOME_ABA_EFEITOS_STOCK = "efeitos_stock"   # mesma planilha de ID_PLANILHA_BIBLIOTECA_MATCH_AUDIO

# Cole aqui o id OU url_preview de cada linha da aba efeitos_stock que você
# quer disponibilizar como candidata -- casa por versículo individual (não
# por trecho de evento como a trilha), e a maioria dos versículos fica sem
# efeito nenhum (é pontual, não cobertura obrigatória).
EFEITOS_CANDIDATOS = [
]
VOLUME_EFEITO = 0.8       # volume de cada efeito sobreposto (relativo ao áudio já mixado)
DIST_MIN_REPETICAO_EFEITO = 2   # não repete o mesmo efeito em versículos a menos de N de distância

# ── 8. VERSÍCULOS ────────────────────────────────────────────────────────
# Cole aqui o texto do capítulo com os números de versículo isolados no meio
# do fluxo (mesmo formato do indicador "Matt 2:4") -- ou deixe vazio pra usar
# o arquivo padronizado já salvo em videos/<nome>/ (ver limpar_roteiro_biblia.html)
TEXTO_VERSICULOS = """"""


print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Vídeo:              {NOME_ORACAO}")
print(f"   Capítulo:           {LIVRO_PT} {CAPITULO}")
print(f"   Fonte da cena:      {TIPO_FONTE}")
print(f"   Voz:                {VOZ_EDGE}")
print(f"   Volume narr./trilha: {VOLUME_NARRACAO} / {VOLUME_MUSICA}")
print(f"   Velocidade:         {VELOCIDADE_AUDIO}x")
print(f"   Léxico:             {'ON' if USAR_LEXICO_BIBLICO else 'off (100% IA)'}")
print(f"   Biblioteca de match: {'ON' if USAR_BIBLIOTECA_MATCH else 'off'}")
print(f"   Trilhas candidatas: {len(TRILHAS_CANDIDATAS)}")
print(f"   Efeitos candidatos: {len(EFEITOS_CANDIDATOS)}")
print(f"   Versículos:         {'(vazio! preencha TEXTO_VERSICULOS ou use o arquivo do Drive)' if not TEXTO_VERSICULOS.strip() else 'preenchido'}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📄 ROTEIRO E ÁUDIO DO DRIVE (opcional)                          ║
# ║  Se existirem no Drive, SUBSTITUEM TEXTO_ORACAO e pulam a narração ║
# ║  Roteiro: [NOME]_roteiro.txt                                     ║
# ║  Áudio:   [NOME]_audio.wav (ou .mp3/.m4a/.ogg — convertido sozinho) ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil, subprocess

PASTA_VIDEO_DRIVE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}")
PASTA_VIDEO_DRIVE.mkdir(parents=True, exist_ok=True)

# ── Roteiro (texto) ─────────────────────────────────────────────────────
roteiro_drive = PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_roteiro.txt"
if roteiro_drive.exists():
    TEXTO_ORACAO = roteiro_drive.read_text(encoding="utf-8").strip()
    print(f"✅ Roteiro carregado do Drive: {roteiro_drive.name} ({len(TEXTO_ORACAO)} caracteres)")
else:
    print(f"ℹ️  Sem roteiro em {roteiro_drive.name} — usando o texto da célula de Configuração")

# ── Áudio — aceita .wav, .mp3, .m4a, .ogg; converte pra wav se precisar ────
audio_local = Path(f"/content/{NOME_ORACAO}_audio.wav")
EXTENSOES_AUDIO = [".wav", ".mp3", ".m4a", ".ogg", ".flac"]

audio_drive = next(
    (PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_audio{ext}" for ext in EXTENSOES_AUDIO
     if (PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_audio{ext}").exists()),
    None,
)

if audio_drive is None:
    print(f"ℹ️  Sem áudio em {PASTA_VIDEO_DRIVE.name}/{NOME_ORACAO}_audio.* — vai ser gerado pelo Edge TTS")
elif audio_drive.suffix == ".wav":
    shutil.copy2(audio_drive, audio_local)
    print(f"✅ Áudio (.wav) carregado do Drive: {audio_drive.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
else:
    print(f"🔄 Áudio encontrado como {audio_drive.suffix}: {audio_drive.name} — convertendo pra .wav...")
    resultado = subprocess.run(
        ["ffmpeg", "-y", "-i", str(audio_drive), "-ar", "44100", "-ac", "2", str(audio_local)],
        capture_output=True, text=True,
    )
    if audio_local.exists():
        print(f"✅ Convertido: {audio_local.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
        shutil.copy2(audio_local, PASTA_VIDEO_DRIVE / audio_local.name)
    else:
        print(f"❌ Conversão falhou — confira o arquivo original")
        print(resultado.stderr[-800:])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ BAIXAR ÁUDIO DUBLADO (YouTube, opcional)                    ║
# ║                                                                    ║
# ║  Puxa uma faixa de dublagem automática de um vídeo do YouTube (via ║
# ║  yt-dlp) em vez de gerar a narração com Edge TTS.                 ║
# ║                                                                    ║
# ║  Salva como [NOME]_audio.wav — a célula de NARRAÇÃO abaixo detecta ║
# ║  que já existe e pula a geração automaticamente.                 ║
# ║                                                                    ║
# ║  TOTALMENTE OPCIONAL — pule esta célula pra usar o Edge TTS normal. ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from pathlib import Path

URL_DUBLAGEM    = "https://www.youtube.com/watch?v=4vTN7tBG3a8"  # url do vídeo com dublagem automática
IDIOMA_DUBLAGEM = "en"    # código do idioma da faixa que você quer (ex: "pt", "en", "es", "fr")
FORMAT_ID_MANUAL = ""     # deixe vazio pra tentar automático; se falhar, veja a lista impressa
                          # abaixo e cole o ID exato da faixa aqui (ex: "233-1")

audio_local = Path(f"/content/{NOME_ORACAO}_audio.wav")

print("📋 Faixas de áudio disponíveis nesse vídeo:")
lista = subprocess.run(["yt-dlp", "-F", URL_DUBLAGEM], capture_output=True, text=True)
for linha in lista.stdout.splitlines():
    if "audio only" in linha:
        print("  ", linha)
print()

formato = FORMAT_ID_MANUAL.strip() or f"ba[language^={IDIOMA_DUBLAGEM}]/bestaudio[language^={IDIOMA_DUBLAGEM}]"
print(f"🎙️  Baixando faixa de áudio (formato: {formato})...")

resultado = subprocess.run(
    ["yt-dlp", "-f", formato, "--extract-audio", "--audio-format", "wav",
     "-o", "temp_dublagem.%(ext)s", URL_DUBLAGEM],
    capture_output=True, text=True,
)

temp_wav = Path("temp_dublagem.wav")
if temp_wav.exists():
    temp_wav.replace(audio_local)
    print(f"✅ Áudio dublado salvo: {audio_local.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
else:
    print("❌ Não achei uma faixa correspondente automaticamente.")
    print("   Confira a lista impressa acima, copie o ID da faixa que quer")
    print("   (coluna da esquerda, ex: '233-1') e cole em FORMAT_ID_MANUAL. Detalhe do erro:")
    print(resultado.stderr[-800:])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR PIPELINE                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint

MODO_CLIPE = "imagem" if TIPO_FONTE == "imagem" else "video"

config = PipelineConfig(
    NOME_ORACAO                  = NOME_ORACAO,
    PASTA_DRIVE_RAIZ              = PASTA_DRIVE_RAIZ,
    TEXTO_ORACAO                  = TEXTO_ORACAO,
    VOZ_EDGE                      = VOZ_EDGE,
    IDIOMA_MESTRE                 = IDIOMA_MESTRE,
    VOLUME_NARRACAO               = VOLUME_NARRACAO,
    VOLUME_MUSICA                 = VOLUME_MUSICA,
    VELOCIDADE_AUDIO              = VELOCIDADE_AUDIO,
    MODO_CLIPE                    = MODO_CLIPE,
    ID_PLANILHA_IMAGENS_DRIVE     = ID_PLANILHA_IMAGENS,
    NOME_COLUNA_STATUS_PLANILHA   = NOME_COLUNA_STATUS_PLANILHA,
)

pipeline = VideoPipeline(config)
cp       = Checkpoint(nome_oracao=config.NOME_ORACAO)  # isolado por vídeo -- ver checkpoint.py

from drive_utils import DriveClient
_drive = DriveClient.get()  # singleton -- mesma instância que o pipeline usa internamente

print("=" * 60)
print("✅ PIPELINE INICIALIZADO")
print("=" * 60)
print(f"   Vídeo:       {config.NOME_ORACAO}")
print(f"   Pasta:       {config.pasta_oracao}")
print(f"   Checkpoint:  {cp.proxima_fase_pendente() or 'tudo feito'}")
print("=" * 60)
print(config.resumo())


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎤 NARRAÇÃO — gera o áudio com Edge TTS                        ║
# ║  Pula a geração se o áudio já existir (Drive/manual/etc.)        ║
# ╚══════════════════════════════════════════════════════════════════╝

audio = pipeline.gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 MATCH CENA↔VERSÍCULO — carregar legenda mestre + biblioteca  ║
# ║  (incorporado de match-scene-verse.ipynb -- reusa o `config` e   ║
# ║  o `pipeline` já inicializados acima, não cria um config novo)   ║
# ╚══════════════════════════════════════════════════════════════════╝
from srt_utils import ler_srt, texto_por_versiculo
from match_pipeline import (
    carregar_biblioteca, gerar_sugestoes_match, carregar_lexico_biblico,
    abrir_ou_criar_biblioteca_match, carregar_biblioteca_match,
    garantir_aba_versiculo_tags, carregar_versiculo_tags,
    garantir_aba_evento_tags, garantir_aba_titulo_tags,
    carregar_evento_tags, carregar_titulo_tags,
)

# 1. legenda mestre (Whisper bruto) -- baixa do Drive se não estiver local
_caminho_srt_mestre = Path(config.nome_srt_edge(IDIOMA_MESTRE))
if not _caminho_srt_mestre.exists():
    _drive.download(config.pasta_oracao, config.nome_srt_edge(IDIOMA_MESTRE), _caminho_srt_mestre)
if not _caminho_srt_mestre.exists():
    raise FileNotFoundError(
        f"{_caminho_srt_mestre.name} não encontrado (local nem no Drive) — "
        "rode caption-single-generate.ipynb primeiro (ver célula de introdução)."
    )
legendas_mestre = ler_srt(_caminho_srt_mestre)
print(f"✅ Legenda mestre: {len(legendas_mestre)} blocos ({_caminho_srt_mestre.name})")

# 2. texto de cada versículo -- tenta o arquivo padronizado no Drive primeiro
if not TEXTO_VERSICULOS.strip():
    _dest_roteiro_versiculos = Path(config.nome_roteiro_versiculos)
    _drive.download_se_ausente(config.pasta_oracao, config.nome_roteiro_versiculos, _dest_roteiro_versiculos)
    if _dest_roteiro_versiculos.exists():
        TEXTO_VERSICULOS = _dest_roteiro_versiculos.read_text(encoding="utf-8")
        print(f"✅ TEXTO_VERSICULOS carregado do arquivo ({config.nome_roteiro_versiculos})")
    else:
        raise ValueError(
            f"TEXTO_VERSICULOS está vazio e {config.nome_roteiro_versiculos} não foi encontrado "
            f"(local nem no Drive) -- cole o texto na célula de Configuração, ou gere o arquivo "
            f"com limpar_roteiro_biblia.html e suba pra videos/{NOME_ORACAO}/."
        )
versiculos_texto = texto_por_versiculo(TEXTO_VERSICULOS)
print(f"✅ {len(versiculos_texto)} versículos extraídos")

# 3. abre a planilha certa conforme TIPO_FONTE
if TIPO_FONTE == "video":
    id_planilha, nome_aba, coluna_url, coluna_tags_biblia = ID_PLANILHA_VIDEOS, NOME_ABA_VIDEOS, "url", "Tags_Biblia_PT"
elif TIPO_FONTE == "imagem":
    id_planilha, nome_aba, coluna_url, coluna_tags_biblia = ID_PLANILHA_IMAGENS, NOME_ABA_IMAGENS, "Imagem", "Tags_Semelhantes_PT"
else:
    raise ValueError(f"TIPO_FONTE inválido: {TIPO_FONTE!r} (use 'video' ou 'imagem')")

sheet = gc.open_by_key(id_planilha).worksheet(nome_aba)
linhas_planilha = sheet.get_all_records()
biblioteca = carregar_biblioteca(linhas_planilha, coluna_url=coluna_url, coluna_tags_biblia=coluna_tags_biblia)
print(f"✅ Biblioteca ({TIPO_FONTE}): {len(linhas_planilha)} linhas na planilha, "
      f"{len(biblioteca)} já têm {coluna_tags_biblia} preenchida (só essas entram no match)")

# 4. léxico bíblico (opcional, ver USAR_LEXICO_BIBLICO)
titulos_biblicos = eventos_biblicos = None
if USAR_LEXICO_BIBLICO:
    _dest_titulos = Path(NOME_ARQUIVO_TITULOS)
    _dest_eventos = Path(NOME_ARQUIVO_EVENTOS)
    _drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_TITULOS, _dest_titulos)
    _drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_EVENTOS, _dest_eventos)
    if _dest_titulos.exists() and _dest_eventos.exists():
        titulos_biblicos, eventos_biblicos = carregar_lexico_biblico(_dest_titulos, _dest_eventos)
        print(f"✅ Léxico bíblico: {len(titulos_biblicos)} títulos, {len(eventos_biblicos)} eventos")
    else:
        print(f"⚠️  Léxico não encontrado em {PASTA_DADOS_LEXICO} (local nem Drive) -- caindo pra 100% IA")

# 5. biblioteca de match (cache entre vídeos)
biblioteca_match = {}
aba_biblioteca_match = None
_spreadsheet_biblioteca = None
if USAR_BIBLIOTECA_MATCH:
    _spreadsheet_biblioteca, aba_biblioteca_match, _id_usado = abrir_ou_criar_biblioteca_match(
        gc, ID_PLANILHA_BIBLIOTECA_MATCH, NOME_ABA_BIBLIOTECA_MATCH,
    )
    biblioteca_match = carregar_biblioteca_match(aba_biblioteca_match)
    print(f"✅ Biblioteca de match: {len(biblioteca_match)} título(s)/evento(s) já resolvido(s) antes")

# 6. tags por versículo
aba_versiculo_tags = None
if USAR_VERSICULO_TAGS:
    if not USAR_BIBLIOTECA_MATCH:
        print("⚠️  USAR_VERSICULO_TAGS precisa de USAR_BIBLIOTECA_MATCH=True (usa a mesma planilha) -- ignorando.")
    else:
        aba_versiculo_tags = garantir_aba_versiculo_tags(_spreadsheet_biblioteca, NOME_ABA_VERSICULO_TAGS)
        print(f"✅ Tags por versículo: aba pronta ({len(carregar_versiculo_tags(aba_versiculo_tags))} versículo(s) já taggeado(s))")

# 7. evento_tags/titulo_tags (fonte editável na planilha -- também usados pela TRILHA mais abaixo)
aba_evento_tags = garantir_aba_evento_tags(_spreadsheet_biblioteca)
aba_titulo_tags = garantir_aba_titulo_tags(_spreadsheet_biblioteca)
evento_tags_dict = carregar_evento_tags(aba_evento_tags)
titulo_tags_dict = carregar_titulo_tags(aba_titulo_tags)
print(f"✅ evento_tags/titulo_tags carregados da planilha ({len(evento_tags_dict)} evento(s), {len(titulo_tags_dict)} título(s))")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📋 LISTA FECHADA DE TAGS_BIBLIA                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
LISTA_TAGS_BIBLIA = ("criação, jardim do éden, queda, dilúvio, arca de noé, torre de babel, "
"chamado de abraão, aliança, sacrifício de isaque, jacó e esaú, escada de jacó, josé e os "
"irmãos, sonhos proféticos, escravidão no egito, sarça ardente, pragas do egito, travessia do "
"mar vermelho, maná no deserto, dez mandamentos, monte sinai, bezerro de ouro, tabernáculo, "
"arca da aliança, peregrinação no deserto, serpente de bronze, terra prometida, queda de "
"jericó, juízes, sansão, gideão, débora, rute e noemi, davi e golias, davi e saul, reino de "
"davi, sabedoria de salomão, templo de salomão, reino dividido, elias no monte carmelo, carro "
"de fogo, eliseu, exílio babilônico, daniel na cova dos leões, fornalha ardente, jonas e o "
"grande peixe, ester, sofrimento de jó, salmos e louvor, provérbios e sabedoria, reconstrução "
"do templo, profecia messiânica, anunciação, natividade, magos do oriente, estrela de belém, "
"fuga para o egito, apresentação no templo, batismo de jesus, tentação no deserto, chamado dos "
"discípulos, sermão da montanha, bem-aventuranças, milagre de cura, multiplicação dos pães, "
"tempestade acalmada, jesus anda sobre as águas, parábola do semeador, parábola do filho "
"pródigo, parábola do bom samaritano, ovelha perdida, transfiguração, ressurreição de lázaro, "
"entrada triunfal em jerusalém, última ceia, getsêmani, prisão e julgamento, crucificação, "
"ressurreição de jesus, tumba vazia, estrada de emaús, ascensão, pentecostes, conversão de "
"paulo, viagens missionárias, igreja primitiva, perseguição dos cristãos, cartas apostólicas, "
"apocalipse, pastor e ovelhas, boas novas, anjo mensageiro, profeta, rei, sacerdote, juízo, "
"misericórdia divina, aliança renovada, êxodo espiritual, batalha espiritual, jornada de fé, "
"provação, milagre, cura, ressurreição, segunda vinda, reino de deus, cordeiro de deus, luz "
"do mundo")

print(f"{len(LISTA_TAGS_BIBLIA.split(','))} temas na lista fechada")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎯 RODAR (OU REAPROVEITAR) O MATCH CENA↔VERSÍCULO               ║
# ║  Se já existe um match salvo pra este vídeo/capítulo (local ou    ║
# ║  Drive), reaproveita sem gastar IA de novo. Pra forçar um match   ║
# ║  novo, apague match_<nome>_cap<N>.json local E no Drive antes.    ║
# ╚══════════════════════════════════════════════════════════════════╝
import json as _json

_caminho_match = Path(config.nome_match_json(CAPITULO))
_drive.download_se_ausente(config.pasta_oracao, config.nome_match_json(CAPITULO), _caminho_match)

if _caminho_match.exists():
    with open(_caminho_match, encoding="utf-8") as _f:
        resultados_match = _json.load(_f)
    print(f"♻️  Match já existia ({_caminho_match.name}) -- reaproveitando, {len(resultados_match)} versículo(s).")
else:
    if not (groq_client or mistral_client):
        print("⚠️  Nenhuma API de IA disponível (GROQ_KEY/MISTRAL_KEY) -- só versículos já cobertos "
              "pela biblioteca de match ou pelo léxico vão resolver; o resto fica 'sem opção'.")

    resultados_match = gerar_sugestoes_match(
        versiculos_texto, biblioteca, LISTA_TAGS_BIBLIA,
        groq_client, mistral_client, MODELO_GROQ, MODELO_MISTRAL,
        dist_min_repeticao=DIST_MIN_REPETICAO, margem_para_repetir=MARGEM_PARA_REPETIR,
        delay_segundos=DELAY_SEGUNDOS, max_tokens=MAX_TOKENS_RESPOSTA,
        livro_pt=LIVRO_PT if USAR_LEXICO_BIBLICO else None, capitulo=CAPITULO,
        titulos_biblicos=titulos_biblicos, eventos_biblicos=eventos_biblicos,
        tipo_fonte=TIPO_FONTE, biblioteca_match=biblioteca_match, aba_biblioteca_match=aba_biblioteca_match,
        aba_versiculo_tags=aba_versiculo_tags,
    )

    with open(_caminho_match, "w", encoding="utf-8") as f:
        _json.dump(resultados_match, f, ensure_ascii=False, indent=2)
    _drive.upload(_caminho_match, config.pasta_oracao, "application/json")
    print(f"☁️  Match salvo em {_caminho_match.name} e enviado pro Drive")

com_match = sum(1 for r in resultados_match if not r["sem_opcao"])
print(f"\n{'='*60}")
print(f"✅ {len(resultados_match)} versículos processados -- com sugestão: {com_match}, sem opção: {len(resultados_match) - com_match}")
print(f"{'='*60}")
for r in resultados_match:
    cap_v = f"{CAPITULO}:{r['versiculo']}"
    if r["sem_opcao"]:
        print(f"❌ {cap_v:6s} SEM OPÇÃO — busque por: {', '.join(r.get('palavras_chave', [])) or '(nenhuma sugestão)'}")
    else:
        emoji_fonte = {"biblioteca": "🗃️", "lexico": "📚"}.get(r.get("fonte"), "🤖")
        print(f"✅ {cap_v:6s} {emoji_fonte} [{r['id']}] {r['titulo']}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🖼️🏷️ CONVERTER CENAS + CRÉDITO + LOGO                           ║
# ║  Usa o resultado do match acima (em memória) -- exige cobertura   ║
# ║  100% (nenhum versículo "sem opção"), senão para com um relatório ║
# ║  do que falta buscar.                                             ║
# ╚══════════════════════════════════════════════════════════════════╝
from srt_utils import alinhar_versiculos
from match_pipeline import verificar_cobertura_match, calcular_segmentos_versiculo, PipelineNaoCoberto
from ffmpeg_utils import obter_duracao

# 1. cobertura 100% -- para aqui com relatório se faltar algum
try:
    verificar_cobertura_match(resultados_match, config.nome_lacunas_match(CAPITULO))
except PipelineNaoCoberto as _erro:
    print(f"\n❌ {_erro}")
    print(f"\n💾 Lista salva em {config.nome_lacunas_match(CAPITULO)} — busque essas palavras-chave, adicione a "
          "URL + Tags na planilha, apague o match_*.json local e no Drive, e rode a célula do match de novo.")
    raise

# 2. tempo de início de cada versículo
tempos_versiculo = alinhar_versiculos(TEXTO_VERSICULOS, legendas_mestre)

# 3. duração total da narração
_audio_path = Path(config.NOME_AUDIO)
_drive.download_se_ausente(config.pasta_assets_audio, config.NOME_AUDIO, _audio_path)
duracao_total_ms = obter_duracao(_audio_path) * 1000

# 4. plano de segmentos (funde versículos curtos com o vizinho)
plano_segmentos = calcular_segmentos_versiculo(
    resultados_match, tempos_versiculo, duracao_total_ms,
    config.DURACAO_MINIMA_SEGMENTO_VERSICULO,
)
print(f"📋 {len(plano_segmentos)} segmento(s) planejado(s) a partir de {len(tempos_versiculo)} versículos "
      f"(mín. {config.DURACAO_MINIMA_SEGMENTO_VERSICULO}s/segmento)")

if TIPO_FONTE == "imagem":
    clipes = pipeline.baixar_clipes_imagem_por_versiculo(plano_segmentos)
else:
    clipes = pipeline.baixar_clipes_por_versiculo(plano_segmentos)

print(f"\n✅ {len(clipes)} segmento(s) pronto(s) em clipes_cortados/ (versículo por versículo)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💾 SALVAR CENAS NO DRIVE (opcional)                             ║
# ║  Copia as cenas já cortadas/creditadas deste vídeo pro pool       ║
# ║  compartilhado (assets/clipes/), pra vídeos futuros reaproveitarem. ║
# ║  TOTALMENTE OPCIONAL — só funciona (e faz sentido) pra TIPO_FONTE ║
# ║  == "video"; em modo imagem não há pool compartilhado.            ║
# ╚══════════════════════════════════════════════════════════════════╝
if TIPO_FONTE == "video":
    n_salvos = pipeline.salvar_clipes_no_drive(clipes)
    print(f"💾 {n_salvos} clipe(s) novo(s) salvo(s) em {config.pasta_assets_clipes}")
else:
    print("ℹ️  Modo imagem não usa pool compartilhado -- nada a salvar aqui.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎵 TRILHA — match por clima (por evento) + montagem sequencial  ║
# ║                                                                    ║
# ║  Segundo match, independente do de cena: agrupa versículos        ║
# ║  consecutivos do MESMO evento bíblico sob a MESMA trilha (o clima ║
# ║  muda quando a cena muda, não verso a verso). Escolhe entre as    ║
# ║  trilhas candidatas coladas em TRILHAS_CANDIDATAS (Configuração), ║
# ║  pelo clima do TÍTULO do versículo (mais específico) ou, se não   ║
# ║  tiver, do EVENTO inteiro. Trecho sem clima cadastrado ou sem     ║
# ║  nenhuma candidata batendo fica em SILÊNCIO (não é erro -- ver o  ║
# ║  relatório de lacunas abaixo).                                    ║
# ╚══════════════════════════════════════════════════════════════════╝
from trilha_pipeline import (
    carregar_trilha_stock_da_planilha, carregar_evento_clima, carregar_titulo_clima,
    calcular_segmentos_trilha, relatorio_lacunas_trilha, baixar_trilha,
)
from ffmpeg_utils import montar_trilha_sequencial

_aba_trilha_stock = gc.open_by_key(ID_PLANILHA_BIBLIOTECA_MATCH_AUDIO).worksheet(NOME_ABA_TRILHA_STOCK)
_trilha_stock_completa = carregar_trilha_stock_da_planilha(_aba_trilha_stock)
trilha_pool = [t for t in _trilha_stock_completa if t["id"] in TRILHAS_CANDIDATAS or t["url"] in TRILHAS_CANDIDATAS]
print(f"🎵 Pool de trilhas: {len(trilha_pool)}/{len(TRILHAS_CANDIDATAS)} candidata(s) encontrada(s) na trilha_stock")
if len(trilha_pool) < len(TRILHAS_CANDIDATAS):
    print("   ⚠️  alguma(s) candidata(s) não foi encontrada -- confira os ids/urls colados em TRILHAS_CANDIDATAS")

evento_clima_dict = carregar_evento_clima(aba_evento_tags)
titulo_clima_dict = carregar_titulo_clima(aba_titulo_tags)

plano_trilha = calcular_segmentos_trilha(
    versiculos_texto, tempos_versiculo, duracao_total_ms,
    LIVRO_PT, CAPITULO, titulos_biblicos, eventos_biblicos,
    evento_clima_dict, trilha_pool, titulo_clima_dict=titulo_clima_dict,
)
print(f"\n📋 {len(plano_trilha)} segmento(s) de trilha planejado(s):")
for seg in plano_trilha:
    v_ini, v_fim = seg["versiculos"][0], seg["versiculos"][-1]
    faixa = f"v{v_ini}" if v_ini == v_fim else f"v{v_ini}-{v_fim}"
    if seg["trilha"]:
        print(f"   {faixa:>10s}  ({seg['duracao_seg']:5.1f}s)  🎵 {seg['trilha']['titulo']}  (score={seg['trilha']['score']})")
    else:
        print(f"   {faixa:>10s}  ({seg['duracao_seg']:5.1f}s)  🔇 sem trilha (evento: {seg['titulo_evento'] or '?'})")

_lacunas = relatorio_lacunas_trilha(plano_trilha)
if _lacunas:
    print(f"\n⚠️  {_lacunas}")

# baixa cada trilha ÚNICA escolhida (evita baixar a mesma trilha 2x se ela
# cobrir vários trechos não-consecutivos do vídeo)
Path("trilhas_baixadas").mkdir(exist_ok=True)
_arquivo_por_trilha_id = {}
for seg in plano_trilha:
    if not seg["trilha"]:
        continue
    _tid = seg["trilha"]["id"]
    if _tid in _arquivo_por_trilha_id:
        continue
    _destino = Path(f"trilhas_baixadas/{_tid}.mp3")
    _url = seg["trilha"].get("url") or ""
    if baixar_trilha(_url, _destino):
        _arquivo_por_trilha_id[_tid] = _destino
        print(f"   ✅ baixada: {seg['trilha']['titulo']}")
    else:
        _arquivo_por_trilha_id[_tid] = None
        print(f"   ❌ falha ao baixar: {seg['trilha']['titulo']} ({_url})")

segmentos_montagem = [
    {"arquivo": _arquivo_por_trilha_id.get(seg["trilha"]["id"]) if seg["trilha"] else None,
     "duracao_seg": seg["duracao_seg"]}
    for seg in plano_trilha
]

trilha_montada = montar_trilha_sequencial(segmentos_montagem, Path(f"{NOME_ORACAO}_trilha_montada.wav"))
print(f"\n✅ Trilha montada: {trilha_montada.name} ({trilha_montada.stat().st_size/1024:.0f} KB)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 VÍDEO BASE — concatena + narração + trilha montada           ║
# ╚══════════════════════════════════════════════════════════════════╝
from pathlib import Path
from ffmpeg_utils import obter_duracao

video_base = pipeline.criar_video_base(clipes, trilha_path=trilha_montada)

if video_base and video_base.exists():
    duracao_final = obter_duracao(video_base)
    duracao_audio = obter_duracao(Path(config.NOME_AUDIO))
    print(f"✅ VÍDEO BASE: {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB, {duracao_final:.1f}s)")
    if abs(duracao_final - duracao_audio) > 1.0:
        print(f"⚠️  Aviso: vídeo ({duracao_final:.1f}s) não bate com o áudio ({duracao_audio:.1f}s) -- confira.")
    else:
        print(f"✅ Duração bate com o áudio ({duracao_audio:.1f}s)")
else:
    print("⚠️  Vídeo base não foi gerado -- confira os logs acima.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💥 EFEITOS SONOROS — match pontual + sobreposição                ║
# ║                                                                    ║
# ║  Terceiro match, independente de cena e trilha: usa as mesmas     ║
# ║  palavras-chave por versículo já calculadas no match de cena      ║
# ║  (tags_semelhantes) contra as candidatas em EFEITOS_CANDIDATOS.   ║
# ║  Casa por VERSÍCULO INDIVIDUAL (não por trecho de evento) -- a    ║
# ║  maioria fica sem efeito nenhum, de propósito. Sobrepõe cada      ║
# ║  efeito escolhido no timestamp exato do versículo, por cima do    ║
# ║  áudio já mixado (narração + trilha) do vídeo base.                ║
# ║                                                                    ║
# ║  IMPORTANTE: o resultado SOBRESCREVE config.NOME_VIDEO_BASE (local ║
# ║  e no Drive) -- é o mesmo nome canônico que caption-*.ipynb já     ║
# ║  espera, então as legendas pegam a versão com efeito automatica-   ║
# ║  mente depois, sem precisar configurar nada de novo lá.            ║
# ╚══════════════════════════════════════════════════════════════════╝
from match_pipeline import carregar_versiculo_tags
from trilha_pipeline import carregar_efeitos_stock_da_planilha, calcular_efeitos_pontuais, baixar_trilha
from ffmpeg_utils import adicionar_efeitos_pontuais

if not EFEITOS_CANDIDATOS:
    print("ℹ️  EFEITOS_CANDIDATOS está vazio -- nenhum efeito sonoro será adicionado.")
    print(f"   Vídeo base permanece: {video_base.name} (já é config.NOME_VIDEO_BASE -- nada a fazer)")
else:
    _aba_efeitos_stock = gc.open_by_key(ID_PLANILHA_BIBLIOTECA_MATCH_AUDIO).worksheet(NOME_ABA_EFEITOS_STOCK)
    _efeitos_stock_completa = carregar_efeitos_stock_da_planilha(_aba_efeitos_stock)
    efeitos_pool = [e for e in _efeitos_stock_completa if e["id"] in EFEITOS_CANDIDATOS or e["url"] in EFEITOS_CANDIDATOS]
    print(f"💥 Pool de efeitos: {len(efeitos_pool)}/{len(EFEITOS_CANDIDATOS)} candidata(s) encontrada(s) na efeitos_stock")
    if len(efeitos_pool) < len(EFEITOS_CANDIDATOS):
        print("   ⚠️  alguma(s) candidata(s) não foi encontrada -- confira os ids/urls colados em EFEITOS_CANDIDATOS")

    versiculo_tags_dict = carregar_versiculo_tags(aba_versiculo_tags) if aba_versiculo_tags is not None else {}

    pontos_efeito = calcular_efeitos_pontuais(
        tempos_versiculo, LIVRO_PT, CAPITULO, versiculo_tags_dict, efeitos_pool,
        dist_min_repeticao=DIST_MIN_REPETICAO_EFEITO,
    )
    print(f"\n📋 {len(pontos_efeito)} versículo(s) com efeito sonoro casado (de {len(tempos_versiculo)} no total):")
    for p in pontos_efeito:
        print(f"   v{p['versiculo']:<4d} ({p['inicio_ms']/1000:6.1f}s)  💥 {p['efeito']['titulo']}  (score={p['efeito']['score']})")

    if not pontos_efeito:
        print("\nℹ️  Nenhum versículo casou com efeito nenhum -- vídeo base permanece sem alteração.")
    else:
        # baixa cada efeito ÚNICO escolhido (evita baixar o mesmo efeito 2x)
        Path("efeitos_baixados").mkdir(exist_ok=True)
        _arquivo_por_efeito_id = {}
        for p in pontos_efeito:
            _eid = p["efeito"]["id"]
            if _eid in _arquivo_por_efeito_id:
                continue
            _destino = Path(f"efeitos_baixados/{_eid}.mp3")
            _url = p["efeito"].get("url") or ""
            if baixar_trilha(_url, _destino):
                _arquivo_por_efeito_id[_eid] = _destino
                print(f"   ✅ baixado: {p['efeito']['titulo']}")
            else:
                _arquivo_por_efeito_id[_eid] = None
                print(f"   ❌ falha ao baixar: {p['efeito']['titulo']} ({_url})")

        efeitos_para_aplicar = [
            {"inicio_ms": p["inicio_ms"], "arquivo": _arquivo_por_efeito_id.get(p["efeito"]["id"])}
            for p in pontos_efeito
        ]

        _video_com_efeitos_temp = adicionar_efeitos_pontuais(
            video_base, efeitos_para_aplicar, Path("_temp_video_base_efeitos.mp4"),
            volume_efeito=VOLUME_EFEITO,
        )
        # substitui o vídeo base "canônico" (config.NOME_VIDEO_BASE, mesmo
        # arquivo que criar_video_base() já tinha gravado ali) pelo resultado
        # COM efeitos -- local e no Drive. Assim caption-*.ipynb continuam
        # funcionando sem saber (nem precisar saber) que passou por aqui.
        _video_com_efeitos_temp.replace(video_base)
        _drive.upload(video_base, config.pasta_assets_videos, "video/mp4")
        print(f"\n✅ Vídeo base ATUALIZADO com efeitos: {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB)")
        print("   ☁️  Enviado pro Drive (sobrescreve a versão sem efeito) -- caption-*.ipynb pegam essa")
        print("       versão automaticamente, sem nenhuma configuração extra do lado deles.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PRÉVIA DO VÍDEO BASE (com efeitos, se houve algum)           ║
# ╚══════════════════════════════════════════════════════════════════╝
from IPython.display import Video, display

if video_base.exists():
    print(f"🎬 {video_base.name}  ({video_base.stat().st_size/1_048_576:.1f} MB)")
    display(Video(str(video_base), embed=True, width=800))
else:
    print(f"❌ Vídeo não encontrado: {video_base}")
    print("   Rode as células de conversão de cena, vídeo base e efeitos primeiro.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 BAIXAR — VÍDEO BASE (com efeitos, se houve algum)            ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files

if video_base.exists():
    print(f"📥 Baixando {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB)...")
    files.download(str(video_base))
else:
    print(f"❌ Vídeo não encontrado: {video_base}")


### 🔧 Utilitários


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 LIMPEZA SELETIVA                                             ║
# ╚══════════════════════════════════════════════════════════════════╝
from pathlib import Path
import shutil

def limpeza_seletiva():
    print("=" * 60)
    print("🧹 LIMPEZA SELETIVA")
    print("=" * 60)
    print("  1 - 🎵 Áudio (.wav, .mp3)")
    print("  2 - 🎬 Vídeos gerados (base, cenas)")
    print("  3 - 📌 Checkpoint")
    print("  4 - 📁 Pastas temporárias (clipes_cortados/, temp_raw/, trilhas_baixadas/, efeitos_baixados/)")
    print("  5 - 🎵 Trilha montada + match/lacunas salvos localmente")
    print("  6 - 🔥 TUDO (1-5)")
    print("  0 - Cancelar")
    escolha = input("\nDigite os números separados por vírgula: ").strip()
    if escolha == '0':
        return
    opcoes = [int(x.strip()) for x in escolha.split(',')]
    cont = 0

    if 1 in opcoes or 6 in opcoes:
        for f in Path('.').glob('*_audio.wav'):
            f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    if 2 in opcoes or 6 in opcoes:
        for pattern in ['*_video_base*.mp4', 'video_com_audio.mp4', 'video_sem_audio.mp4']:
            for f in Path('.').glob(pattern):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    if 3 in opcoes or 6 in opcoes:
        for cp_file in Path('.').glob('checkpoint*.json'):
            cp_file.unlink(); cont += 1; print(f"   🗑️ {cp_file.name}")

    if 4 in opcoes or 6 in opcoes:
        for pasta in ['clipes_cortados', 'temp_raw', 'trilhas_baixadas', 'efeitos_baixados', '__pycache__']:
            p = Path(pasta)
            if p.exists():
                shutil.rmtree(p); cont += 1; print(f"   🗑️ {pasta}/")

    if 5 in opcoes or 6 in opcoes:
        for pattern in ['*_trilha_montada.wav', '_temp_video_base_efeitos.mp4', 'match_*.json', 'lacunas_match_*.txt']:
            for f in Path('.').glob(pattern):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    print(f"\n✅ {cont} item(ns) removido(s)")

limpeza_seletiva()
